# Паттерн 1: Supervisor — центральный диспетчер

Supervisor — это паттерн мультиагентной системы, в котором один управляющий узел (супервайзер) решает, какого агента вызвать следующим. Супервайзер получает от LLM JSON с именем агента и направляет к нему управление. Цикл продолжается, пока супервайзер не вернёт команду FINISH.

```mermaid
graph LR
    START((START)) --> supervisor{{"🎯 Supervisor<br/><i>маршрутизирует агентов</i>"}}
    supervisor -->|researcher| researcher(["🔹 Researcher<br/><i>ищет факты</i>"])
    supervisor -->|writer| writer(["🔹 Writer<br/><i>пишет текст</i>"])
    supervisor -->|analyst| analyst(["🔹 Analyst<br/><i>анализирует данные</i>"])
    supervisor -->|FINISH| END((END))
    researcher --> supervisor
    writer --> supervisor
    analyst --> supervisor

    classDef coordinator fill:#4A90D9,stroke:#2C5F8A,color:#fff,stroke-width:2px
    classDef worker fill:#2ECC71,stroke:#1A8B4C,color:#fff,stroke-width:2px
    classDef terminal fill:#95A5A6,stroke:#707B7C,color:#fff

    class supervisor coordinator
    class researcher,writer,analyst worker
    class START,END terminal
```

In [1]:
import json
import operator
import sys
from typing import Annotated, TypedDict

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph

sys.path.insert(0, "..")
from llm_config import check_api_key, get_llm

In [2]:
if not check_api_key():
    raise ValueError("API key is not set")
else:
    print("API key is set")

API key is set


In [3]:
@tool
def search_web(query: str) -> str:
    """Поиск информации в интернете через DuckDuckGo."""
    from ddgs import DDGS

    with DDGS() as ddgs:
        results = list(ddgs.text(query, max_results=5))
    if not results:
        return f"По запросу '{query}' ничего не найдено."
    return "\n\n".join(f"- {r['title']}: {r['body']}" for r in results)

## Состояние команды

Состояние `TeamState` содержит пять полей. Ключевое — `results` с редьюсером `operator.add`: каждый агент возвращает список, и LangGraph автоматически добавляет его к существующему, а не перезаписывает. Поле `next_agent` — это способ супервайзера сообщить маршрутизатору, куда направить управление дальше.

In [4]:
llm = get_llm()


class TeamState(TypedDict):
    task: str  # Задача от пользователя
    next_agent: str  # Кого вызвать следующим
    results: Annotated[list[str], operator.add]  # Результаты агентов (аддитивный)
    iteration: int  # Счётчик итераций
    final_answer: str  # Итоговый ответ

## Узел супервайзера

Супервайзер — это LLM с промптом, в котором перечислены доступные агенты. Он анализирует задачу и текущие результаты, а затем возвращает JSON вида `{"next": "agent_name"}`. Критически важен fallback при парсинге JSON: LLM не всегда отвечает чистым JSON, поэтому при ошибке парсинга мы ищем имя агента в тексте ответа.

In [5]:
def supervisor_node(state: TeamState) -> dict:
    """
    Супервайзер: анализирует задачу и текущий прогресс,
    выбирает следующего агента или завершает работу.
    """
    if not state["results"]:
        results_text = "Пока нет результатов."
    else:
        results_text = "\n".join(state["results"])
        
    iteration = state.get("iteration", 0)

    response = llm.invoke(
        [
            SystemMessage(
                content="""Ты — супервайзер команды из трёх агентов:
- researcher: ищет информацию и факты по теме
- writer: пишет связный текст на основе собранных данных
- analyst: анализирует данные, выделяет тренды и делает выводы

Проанализируй задачу и текущие результаты работы команды.
Верни JSON (и ТОЛЬКО JSON, без markdown-блоков) с полем "next":
- "researcher" — если нужно найти информацию
- "writer" — если нужно написать итоговый текст
- "analyst" — если нужен анализ данных
- "FINISH" — если задача полностью выполнена

Типичный порядок: researcher → analyst → writer → FINISH
Если результатов достаточно для ответа — возвращай FINISH."""
            ),
            HumanMessage(
                content=f"Задача: {state['task']}\n\nТекущие результаты:\n{results_text}"
            ),
        ]
    )

    # Парсинг JSON с fallback
    try:
        parsed = json.loads(response.content.strip())
        next_agent = parsed.get("next", "FINISH")
    except json.JSONDecodeError:
        # Fallback: ищем имя агента в тексте
        content = response.content.lower()
        if "researcher" in content:
            next_agent = "researcher"
        elif "writer" in content:
            next_agent = "writer"
        elif "analyst" in content:
            next_agent = "analyst"
        else:
            next_agent = "FINISH"

    print(f"  [Супервайзер, итерация {iteration + 1}] → Следующий: {next_agent}")
    return {"next_agent": next_agent, "iteration": iteration + 1}

## Рабочие узлы

Каждый рабочий агент имеет узкую специализацию и одну задачу. Researcher использует инструмент поиска и LLM для структуризации фактов. Writer создаёт связный текст и записывает его в `final_answer`. Analyst выделяет тренды и закономерности. Все трое возвращают результат через редьюсер `results`.

In [6]:
def researcher_node(state: TeamState) -> dict:
    """Исследователь: ищет ключевые факты по теме."""
    # В реальной системе здесь был бы create_agent с инструментом поиска.
    # Для наглядности вызываем инструмент напрямую + LLM для обработки.
    raw_data = search_web.invoke({"query": state["task"]})

    response = llm.invoke(
        [
            SystemMessage(
                content=(
                    "Ты — исследователь. Систематизируй найденные факты: "
                    "выдели ключевые тезисы, цифры, даты. Не пиши статью — "
                    "только структурированные факты. Кратко, по пунктам."
                )
            ),
            HumanMessage(
                content=f"Задача: {state['task']}\n\nНайденные данные:\n{raw_data}"
            ),
        ]
    )

    result = f"[Исследователь]: {response.content}"
    print(f"  {result[:100]}...")
    return {"results": [result]}


def writer_node(state: TeamState) -> dict:
    """Писатель: создаёт связный текст на основе собранных результатов."""
    results_text = "\n".join(state["results"])

    response = llm.invoke(
        [
            SystemMessage(
                content=(
                    "Ты — талантливый писатель. Напиши связный, увлекательный "
                    "текст (4-5 предложений) на основе предоставленных данных. "
                    "Не выдумывай факты — используй только то, что дали."
                )
            ),
            HumanMessage(
                content=f"Задача: {state['task']}\n\nДанные команды:\n{results_text}"
            ),
        ]
    )

    result = f"[Писатель]: {response.content}"
    print(f"  {result[:100]}...")
    return {"results": [result], "final_answer": response.content}


def analyst_node(state: TeamState) -> dict:
    """Аналитик: анализирует собранные данные, выделяет тренды."""
    results_text = "\n".join(state["results"])

    response = llm.invoke(
        [
            SystemMessage(
                content=(
                    "Ты — аналитик. Проанализируй данные: выдели ключевые тренды, "
                    "закономерности, сделай выводы. Кратко — 3-4 тезиса."
                )
            ),
            HumanMessage(content=f"Задача: {state['task']}\n\nДанные:\n{results_text}"),
        ]
    )

    result = f"[Аналитик]: {response.content}"
    print(f"  {result[:100]}...")
    return {"results": [result]}

## Маршрутизация и сборка графа

Условное ребро из супервайзера направляет управление к нужному рабочему агенту на основе значения `next_agent`. После каждого рабочего — безусловное ребро обратно к супервайзеру, формируя цикл. Параметр `recursion_limit=25` при запуске служит обязательной страховкой от бесконечного зацикливания.

In [7]:
def route_from_supervisor(state: TeamState) -> str:
    next_agent = state.get("next_agent", "FINISH")
    if next_agent == "FINISH":
        return END
    return next_agent


graph = StateGraph(TeamState)
graph.add_node("supervisor", supervisor_node)
graph.add_node("researcher", researcher_node)
graph.add_node("writer", writer_node)
graph.add_node("analyst", analyst_node)

graph.add_edge(START, "supervisor")
graph.add_conditional_edges(
    "supervisor",
    route_from_supervisor,
    {
        "researcher": "researcher",
        "writer": "writer",
        "analyst": "analyst",
        END: END,
    },
)
graph.add_edge("researcher", "supervisor")
graph.add_edge("writer", "supervisor")
graph.add_edge("analyst", "supervisor")

app = graph.compile(checkpointer=MemorySaver())

## Запуск

Типичный порядок работы: researcher (сбор фактов) -> analyst (анализ трендов) -> writer (итоговый текст) -> FINISH. Однако супервайзер может отклониться от этого порядка в зависимости от задачи и промежуточных результатов.

In [9]:
config = {
    "configurable": {"thread_id": "supervisor-full-001"},
    "recursion_limit": 25,
}

result = app.invoke(
    {
        "task": "Подготовь обзор состояния мультиагентных систем в AI",
        "next_agent": "",
        "results": [],
        "iteration": 0,
        "final_answer": "",
    },
    config=config,
)

print(f"\n  Итого итераций супервайзера: {result['iteration']}")
print(f"  Результатов от агентов: {len(result['results'])}")
print(f"  Итоговый ответ: {result.get('final_answer', 'Нет ответа')[:200]}...")

  [Супервайзер, итерация 1] → Следующий: researcher
  [Исследователь]: - **Тема обзора:** состояние мультиагентных систем (МАС) в AI, где несколько AI-аге...
  [Супервайзер, итерация 2] → Следующий: FINISH

  Итого итераций супервайзера: 2
  Результатов от агентов: 1
  Итоговый ответ: ...
